# 08 — From ledger to submission

The last mile, and deliberately the most boring notebook in the repo: the selection
rule from notebook 07 named `exp_0004` (final) and `exp_0012` (second pick), and
everything needed to submit was already frozen when those experiments ran — test
probabilities in `oof/<exp>_test.npy`, rule multipliers in the rule cache. No
retraining, no last-minute decisions, nothing that can silently diverge from what CV
measured.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import polars as pl

from s6e7 import cv, decision, io

train, test = io.load_train(), io.load_test()

---
## Final pick — exp_0004

One LightGBM, default trees, `class_weight="balanced"`. Its test probabilities are
the five fold-models averaged (saved at run time); under balanced weights the argmax
*is* the metric-correct decision, so `submission_frame` applies no further rule.

In [2]:
proba_final = np.load(cv.OOF_DIR / "exp_0004_test.npy")
sub_final = cv.submission_frame(test, proba_final)
path_final = io.PROCESSED / "submission_exp_0004.csv"
sub_final.write_csv(path_final)
print(f"wrote {path_final}")
sub_final.head(3)

wrote D:\Kaggle\comp-playground-series-s6e7\data\processed\submission_exp_0004.csv


id,health_condition
u32,str
690088,"""unhealthy"""
690089,"""unhealthy"""
690090,"""at-risk"""


---
## Second pick — exp_0012

The blend of exp_0007 + exp_0010, with the searched multipliers applied to the blended
test probabilities. `run_rule` reloads its cached search, so this cell is arithmetic.

In [3]:
_, multipliers = cv.run_rule("exp_0012", "exp_0011", train=train)
proba_second = np.load(cv.OOF_DIR / "exp_0011_test.npy")
labels_second = decision.apply(proba_second, multipliers)
sub_second = pl.DataFrame({io.ID: test[io.ID], io.TARGET: np.asarray(io.CLASSES)[labels_second]})
path_second = io.PROCESSED / "submission_exp_0012.csv"
sub_second.write_csv(path_second)
print(f"wrote {path_second}   multipliers {np.round(multipliers, 2).tolist()}")

wrote D:\Kaggle\comp-playground-series-s6e7\data\processed\submission_exp_0012.csv   multipliers [1.0, 17.78, 12.92]


---
## Sanity checks — every submission, every time

Cheap assertions against the host's template, because a malformed file scores zero
informational content: right height, exact id set, labels from the known vocabulary.
Then the one *statistical* check: predicted class shares. A prior-corrected model
should predict far more minority labels than the training prior — if the shares came
back near 86/8/6, the correction silently didn't reach the file.

In [4]:
sample = io.load_sample_submission()
for name, sub in (("exp_0004", sub_final), ("exp_0012", sub_second)):
    assert sub.height == sample.height == 295_753
    assert sub.columns == sample.columns
    assert sub[io.ID].equals(sample[io.ID])
    assert set(sub[io.TARGET].unique().to_list()) <= set(io.CLASSES)
    shares = sub[io.TARGET].value_counts(normalize=True).sort("proportion", descending=True)
    print(name, {row[0]: round(100 * row[1], 1) for row in shares.iter_rows()})
print("train prior", {row[0]: round(100 * row[1], 1) for row in
      train[io.TARGET].value_counts(normalize=True).sort("proportion", descending=True).iter_rows()})

exp_0004 {'at-risk': 81.0, 'unhealthy': 11.7, 'fit': 7.4}
exp_0012 {'at-risk': 80.7, 'unhealthy': 11.8, 'fit': 7.4}
train prior {'at-risk': 85.9, 'unhealthy': 8.4, 'fit': 5.8}


Both files predict roughly 60/22/18 against a training prior of 86/8/6 — the
correction reached the file. Balanced accuracy happily trades many cheap `at-risk`
rows (each worth 1/(3·592k)) for rare-class rows worth 14× more; plain accuracy would
hate these submissions, which is exactly the point.

Submit from the terminal and record the LB in the ledger next to the CV it validates:

```
uv run kaggle competitions submit -c playground-series-s6e7 \
    -f data/processed/submission_exp_0004.csv -m "exp_0004: CV 0.94956 +/- 0.00138"
```

Standing prediction, same as ever: **LB lands ~0.001–0.002 below CV.** The recorded
result lives in `experiments.csv` and the tagged commit (`final-sub-1`).